### Faster Whisper의 레이턴시를 Sample Rate별로 측정함

In [ ]:
import os
import sys

WORKDIR = os.environ["CONTAINER_WORK_DIR"]
os.chdir(WORKDIR)
print(f"Current Python version: {sys.version}")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path

In [ ]:
from sj_ai_utils.asr.whisper_utils import segments_to_text
from sj_utils.audio import segment_audio
from sj_utils.evaluator import TimeChecker
from sj_utils.file.json import JsonSaver
from sj_ai_utils.datasets.esic_v1 import ESICv1Dataset

In [ ]:
DESCRIPTION = """
test
"""

In [ ]:
MODEL_SIZE = "large-v3"
OUTPUT = f"{WORKDIR}/test/performance_test/output/latency_prompt_test/20250817"
ESIC_VAL = f"{WORKDIR}/test/performance_test/esic/data/val.json"  # use in val

In [ ]:
MIN_PROMPT = 0
MAX_PROMPT = 200
STEP_PROMPT = 1

In [ ]:
yaml_saver = JsonSaver(DESCRIPTION)

In [ ]:
esic_val = Path(ESIC_VAL)
output = Path(OUTPUT)

In [ ]:
dataset = ESICv1Dataset.load(esic_val).sample(-1)

In [ ]:
model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")

In [ ]:
for prompt_length in range(MIN_PROMPT, MAX_PROMPT + 1, STEP_PROMPT):
    save_path = output / f"{prompt_length}.json"
    transcribe_time = TimeChecker()
    for _, audio, _ in dataset:
        prompts = []
        for segment in segment_audio(audio, mean=48000, std=0, max_div=0):
            transcribe_time.start()
            prompt =" ".join(prompts[-prompt_length:]) if prompts and prompt_length > 0 else None
            segments, _ = model.transcribe(
                segment,
                language="en",
                word_timestamps=True,
                initial_prompt=prompt,
            )
            prompts.extend(segments_to_text(segments).split(" "))
            transcribe_time.check()
    result = {
        "segment_length": prompt_length,
        "latency": transcribe_time.metric()
    }
    yaml_saver.save(result, save_path)